In [ ]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import cross_val_score
import os
from sklearn.neighbors import NearestNeighbors
import statsmodels.api as sm

#### Helper function for data export

In [ ]:
def export_data(dataframe, filename=""):
    folder = r"\results\pm10"
    os.makedirs(folder, exist_ok=True)
    filename=filename
    dataframe.to_csv(os.path.join(folder, filename),index=False)

#### Read the model output data

In [ ]:
file_path = r"\data\pm10_filtered_df_PM10_XGB.csv"
dask_df = dd.read_csv(file_path)
dfx = dask_df.compute()
df = dfx.copy()
df.describe()

### Function to tune epsilon

In [ ]:
def tune_epsilon(X, y, cross_val_fold):
    n = len(y)
    epsilons = np.arange(1.1, 9, 1)
    scores = [
        np.median(cross_val_score(
            HuberRegressor(epsilon=e, alpha=0.0, max_iter=200),
            X, y, cv=cross_val_fold, scoring='neg_mean_absolute_error', n_jobs=-1
        ))
        for e in epsilons
    ]
    eps = round(epsilons[np.argmax(scores)], 1)
    print("Epsilon:", eps)
    return eps

#### Calculate Global Theta

In [ ]:
def get_global_theta(df, y_col="y_residuals", d_col="d_residuals", alpha=0.05):
    g = df[[y_col, d_col]].replace([np.inf, -np.inf], np.nan).dropna()

    D = g[d_col].astype(float).to_numpy().reshape(-1, 1)
    Y = g[y_col].astype(float).to_numpy()
    model = sm.RLM(Y, D, M=sm.robust.norms.HuberT(t=tune_epsilon(D, Y, 2))).fit()

    theta = model.params[0]
    se = model.bse[0]
    ci_low, ci_high = model.conf_int(alpha=alpha)[0]
    p_value = model.pvalues[0]

    print(
        f"Global theta: {theta:.4f} "
        f"(SE={se:.4f}, "
        f"{int((1-alpha)*100)}% CI: [{ci_low:.4f}, {ci_high:.4f}], "
        f"{p_value:.4f}]"
    )
    return theta, se, ci_low, ci_high

work = df[(df["APCP"] > 0.1)].copy()
theta, se, ci_low, ci_high = get_global_theta(work)

#### Function for Theta estimation

In [ ]:
def estimate_theta(df, threshold=0.1, data_bins=[], tune_epsilon_val=True, default_epsilon=4.5,
                   binning_variable="APCP"):
    work = df.copy()
    cols = ["coeff", "APCP", "prev_pm", "d_residuals", "y_residuals", "Density"]
    work = work.dropna(subset=cols)
    work = work[work["APCP"] > threshold].copy()

    # Detect categorical bins
    categorical_bins = all(isinstance(b, str) for b in data_bins)

    if categorical_bins:
        work["data_bins"] = work[binning_variable]
        work = work[work["data_bins"].isin(data_bins)].copy()
    else:
        bin_edges = np.array(data_bins, dtype=float)
        bin_edges[-1] = bin_edges[-1] + 1e-9

        bin_labels = [f"{edge:g}"for edge in bin_edges[:-1]]

        work["data_bins"] = pd.cut(
            work[binning_variable],
            bins=bin_edges,
            labels=bin_labels,
            include_lowest=True,
            right=False
        )

    results = []
    row_outputs = []

    for bin_name, g in work.groupby("data_bins", observed=True):
        g = g.copy()

        if len(g) < 20:
            continue

        X = g[["d_residuals"]].values
        y = g["y_residuals"].values

        eps = tune_epsilon(X, y, 2) if tune_epsilon_val else default_epsilon
        model = sm.RLM(y, X, M=sm.robust.norms.HuberT(t=eps)).fit()
        theta = model.params[0]
        se = model.bse[0]
        theta_ci_low, theta_ci_high = model.conf_int(alpha=0.05)[0]

        g["pred_pct_change_due_to_precip"] = theta * g["APCP"]
        g["pred_percent_removal"] = g["pred_pct_change_due_to_precip"]
        mean_apcp = g["APCP"].mean()
        mean_pr_ci_low = theta_ci_low * mean_apcp
        mean_pr_ci_high = theta_ci_high * mean_apcp
        results.append({
            "variable_bin": bin_name,
            "n": len(g),
            "theta": theta,
            "standard_error": se,
            "theta_ci_low": theta_ci_low,
            "theta_ci_high": theta_ci_high,
            "median_prev_pm": g["prev_pm"].median(),
            "mean_prev_pm": g["prev_pm"].mean(),
            "median_APCP": g["APCP"].median(),
            "mean_APCP": g["APCP"].mean(),
            "mean_percent_removal": g["pred_percent_removal"].mean(),
            "mean_percent_removal_ci_low": mean_pr_ci_low,
            "mean_percent_removal_ci_high": mean_pr_ci_high,
            "median_percent_removal": g["pred_percent_removal"].median(),
        })
        row_outputs.append(g)

    summary_df = pd.DataFrame(results)
    return summary_df

In [ ]:
work = df.copy()
work = work[
    (work["APCP"] >=0.1)
].copy()
apcp_bins = [0.1, 0.5, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 50]
apcp_removal_df = estimate_theta(work,threshold=0.1, data_bins=apcp_bins, tune_epsilon_val=True, default_epsilon=1.35, binning_variable="APCP")
apcp_removal_df
export_data(apcp_removal_df, "PM10_removal_by_precipitation.csv")

#### Removal by baseline PM bins

In [ ]:
work = df.copy()
pm_bins = [0.5, 5, 7.5, 10, 12.5, 15, 17.5, 20, 22.5, 25, 27.5, 30, 32.5, 35, 37.5, 50]
pm_labels = [f"{l}-{r}" for l, r in zip(pm_bins[:-1], pm_bins[1:])]
work["PM_bin"] = pd.cut(work["prev_pm"], bins=pm_bins, labels=pm_labels, right=False)
work = work.dropna(subset=["PM_bin"])
matched = []
bin_counts = work["PM_bin"].value_counts().sort_values()
ref_bin = bin_counts.index[len(bin_counts) // 2]
## For fair comparision between concentrations same baseline was used for 2.5 and 10
ref = pd.read_csv(r"\data\reference\overall_reference_for_baseline_pm.csv")
k = 2
rng = np.random.default_rng(44)
matched = []
for base_pm in sorted(work["PM_bin"].unique()):
    g = work[work["PM_bin"] == base_pm].copy()

    if len(g) < k:
        continue

    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(g[["APCP"]])

    dist, idx = nn.kneighbors(ref[["APCP"]])

    chosen = rng.integers(0, k, size=len(ref))

    row_idx = idx[np.arange(len(ref)), chosen]
    row_dist = dist[np.arange(len(ref)), chosen]

    m = g.iloc[row_idx].copy()
    m["nn_dist_prev_pm"] = row_dist
    m["matched_rank"] = chosen + 1
    m["base_PM_bin"] = base_pm

    matched.append(m)

matched_df = pd.concat(matched, ignore_index=True)
results_df = estimate_theta(work, threshold=1, data_bins=pm_bins, tune_epsilon_val=True, default_epsilon=1.35, binning_variable="prev_pm")

#### Removal by Smoke Levels

##Bootstrapping inside the Density

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import HuberRegressor

work = df.copy()
work = df[(df["APCP"] >=0.5) & (df["APCP"] < 100)].copy()
work = work.dropna(subset=["Density", "prev_pm", "d_residuals", "y_residuals"])

ref = work[work["Density"] == 3].copy()

matched = []

for density in sorted(work["Density"].unique()):
    g = work[work["Density"] == density].copy()

    if density == 3:
        m = ref.copy()
        m["nn_dist_prev_pm"] = 0
    else:
        k_neighbors = 1
        rng = np.random.default_rng(42)

        k_use = min(k_neighbors, len(g))

        nn = NearestNeighbors(n_neighbors=k_use)
        nn.fit(g[["prev_pm", "APCP"]])

        dist, idx = nn.kneighbors(ref[["prev_pm", "APCP"]])

        choice = rng.integers(0, k_use, size=len(ref))

        chosen_idx = idx[np.arange(len(ref)), choice]
        chosen_dist = dist[np.arange(len(ref)), choice]

        m = g.iloc[chosen_idx].copy()
        m["nn_dist_prev_pm"] = chosen_dist

    matched.append(m)

matched_df = pd.concat(matched, ignore_index=True)

results = []
epsilon_by_density = {}

for density, g in matched_df.groupby("Density", observed=True):
    X = g[["d_residuals"]].values
    y = g["y_residuals"].values

    eps = tune_epsilon(X, y, 6)
    epsilon_by_density[density] = eps

    model = HuberRegressor(
        fit_intercept=False,
        epsilon=eps,
        alpha=0.0,
        max_iter=500
    )

    model.fit(X, y)
    theta = model.coef_[0]

    results.append({
        "Density": density,
        "n": len(g),
        "theta": theta,
        "mean_APCP": g["APCP"].mean(),
        "mean_prev_pm": g["prev_pm"].mean(),
        "mean_removal": theta * g["APCP"].mean()
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
export_data(results_df, "PM10_smoke_theta_overall.csv")
export_data(matched_df, "PM10_smoke_matched_df_overall.csv")

In [ ]:
n_boot = 200
rng = np.random.default_rng(42)

boot_results = []

for density, g in matched_df.groupby("Density", observed=True):

    g = g.copy()

    if len(g) < 20:
        continue

    for b in range(n_boot):

        boot_g = g.sample(
            n=int(len(g)*0.8),
            replace=True,
            random_state=int(rng.integers(0, 1_000_000_000))
        )

        X = boot_g[["d_residuals"]].values
        y = boot_g["y_residuals"].values

        model = HuberRegressor(
            fit_intercept=False,
            epsilon=epsilon_by_density[density],
            alpha=1e-6,
            max_iter=2000,
            tol=1e-6
        )

        model.fit(X, y)
        theta = model.coef_[0]

        boot_results.append({
            "Density": density,
            "boot": b,
            "n": len(boot_g),
            "epsilon": eps,
            "theta": theta,
            "mean_APCP": boot_g["APCP"].mean(),
            "median_APCP": boot_g["APCP"].median(),
            "mean_prev_pm": boot_g["prev_pm"].mean(),
            "median_prev_pm": boot_g["prev_pm"].median(),
            "mean_removal": theta * boot_g["APCP"].mean(),
            "median_removal": theta * boot_g["APCP"].median()
        })

boot_results_df = pd.DataFrame(boot_results)

boot_results_df

In [ ]:
export_data(boot_results_df, "PM10_smoke_bootstrap_results_overall.csv")